# ARGCN 模型展示教程

## 概述

本 notebook 用 PyTorch 复现 source_repos/reaction-gcnn 中 attention 版本的核心推理路径，目标不是“跑通一个黑盒脚本”，而是把模型拆成几个可观察、可验证的计算模块。

这里说的 ARGCN，可以理解为三段式结构：

- 共享的 RelGCN 分子编码器
- 面向三分子集合的 reaction-level attention readout
- 面向条件空间的多标签分类头

它对应原仓库中的三个核心位置：

- source_repos/reaction-gcnn/models/relgcn.py
- source_repos/reaction-gcnn/models/attention_readout.py
- source_repos/reaction-gcnn/train_attention_model.py

## 0. 环境、依赖与路径定位

紧接着的代码单元不做任何建模，它的作用是建立一个可复现的运行上下文。具体来说，它会：

- 导入后续所有单元都要用到的数值计算、深度学习和化学工具包
- 自动向上查找项目根目录，而不是假设 notebook 一定从某个固定目录启动
- 绑定 demo 数据和原仓库字典文件的位置

你可以把这一步理解成“先定义实验坐标系，再开始讨论模型”。如果路径绑定错了，后面所有张量、标签和模型解释都会失去上下文。

In [ ]:
# ====== Step 0: 导入与路径 ======
# 这一单元只做两件事：
# 1. 导入后续会反复使用的科学计算与化学工具包。
# 2. 自动向上寻找项目根目录，避免 notebook 必须从固定工作目录启动。
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem

def find_project_root(start=None):
    # 从当前目录一路向父目录查找，同时包含 teaching_demos 和 source_repos 的目录
    # 就认为是本仓库根目录。这个函数本质上是在做一个“路径锚定”。
    here = Path(start or os.getcwd()).resolve()
    for path in [here, *here.parents]:
        if (path / 'teaching_demos').exists() and (path / 'source_repos').exists():
            return path
    raise RuntimeError('未找到项目根目录')

PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / 'teaching_demos' / '4.reaction_condition_tutorial' / 'ARGCN'
REPO_DIR = PROJECT_ROOT / 'source_repos' / 'reaction-gcnn'
DEMO_DATA = TUTORIAL_DIR / 'data' / 'demo_data.csv'
DICT_PATH = REPO_DIR / 'data' / 'all_dictionaries' / 'suzuki_dict.csv'

# 打印路径，确认 notebook 当前绑定的是哪一份数据和源码仓库。
print('PROJECT_ROOT =', PROJECT_ROOT)
print('DEMO_DATA    =', DEMO_DATA)
print('DICT_PATH    =', DICT_PATH)

PROJECT_ROOT = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
DEMO_DATA    = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/4.reaction_condition_tutorial/ARGCN/data/demo_data.csv
DICT_PATH    = /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/reaction-gcnn/data/all_dictionaries/suzuki_dict.csv


## 1. 构造与数据处理 notebook 一致的 batch

这段代码的目标，是把原始表格数据转换成模型真正会接收的输入对象。对 ARGCN 来说，一条反应最终要被整理成：

- 三张分子图：Reactant1、Reactant2、Product
- 一个多标签目标向量 labels

更具体地说，后面的模型只关心这些张量：

- atom_ids_1 / adjs_1 / mask_1
- atom_ids_2 / adjs_2 / mask_2
- atom_ids_3 / adjs_3 / mask_3
- labels

这里会完成三类关键变换。

第一类变换是条件编码。原始数据里的金属、配体、碱、溶剂、添加剂是文本名称，模型不能直接读取，所以要先映射到统一的全局标签空间。构造出的目标向量是一个 multi-hot 向量：

$$
y \in \{0, 1\}^C, \quad
y_j = 1 \text{ 表示第 } j \text{ 个条件类别在当前反应中出现}
$$

第二类变换是图构造。每个分子会被写成原子序数序列和按键类型分通道的邻接张量：

$$
A^{(r)}_{ij} =
\begin{cases}
1, & \text{若原子 } i, j \text{ 之间存在第 } r \text{ 类化学键} \\
0, & \text{否则}
\end{cases}
$$

第三类变换是 batch 对齐。因为不同分子的原子数不同，所以需要 padding，并配套构造 mask，让后续消息传递和注意力只在真实节点上发生。

读完这个代码单元后，你应该能回答两个问题：一是“标签是怎么从文本条件变成向量的”，二是“一个反应为什么会被表示成三张分子图而不是一张大图”。

In [ ]:
# ====== Step 1: 构造教学 batch ======
# 这一单元把原始反应数据转成模型可直接读取的张量。
# 核心目标是得到 3 个分子图 + 1 个多标签条件向量：
# - Reactant1 图
# - Reactant2 图
# - Product 图
# - y in {0, 1}^C，其中 C 是全部条件类别数
demo_df = pd.read_csv(DEMO_DATA)
suzuki_dict_df = pd.read_csv(DICT_PATH)

# FAMILY_META 定义“条件家族 -> 字典表中的列名”映射。
# 后面会把不同家族映射到一个统一的全局标签空间。
FAMILY_META = {
    'M': ('metal_name', 'metal_bin'),
    'L': ('ligand_name', 'ligand_bin'),
    'B': ('base_name', 'base_bin'),
    'S': ('solvent_name', 'solvent_bin'),
    'A': ('additive_name', 'additive_bin'),
}
RAW_COL_TO_FAMILY = {
    'metal': 'M',
    'ligand': 'L',
    'base': 'B',
    'solvent': 'S',
    'additive': 'A',
}
BOND_TYPE_TO_CHANNEL = {
    Chem.rdchem.BondType.SINGLE: 0,
    Chem.rdchem.BondType.DOUBLE: 1,
    Chem.rdchem.BondType.TRIPLE: 2,
    Chem.rdchem.BondType.AROMATIC: 3,
}

def extract_family_entries(df, family, name_col, bin_col):
    # 从字典表中抽取某一条件家族的合法条目，并把 M1/M2/... 这类编号转成局部下标。
    # local_index = int(bin_id[1:]) - 1
    part = df[[name_col, bin_col]].dropna().copy()
    part[bin_col] = part[bin_col].astype(str)
    part = part[part[bin_col].str.startswith(family)]
    part['local_index'] = part[bin_col].str[1:].astype(int) - 1
    part = part.drop_duplicates(subset=[bin_col]).sort_values('local_index')
    return part

# 把每个家族内部编号压平成统一的全局分类编号。
# 如果某个家族有 K_f 个真实类别，那么它会占据全局向量的一段连续区间。
family_name_to_global = {}
global_index_to_name = {}
total_real_slots = 0
for family, (name_col, bin_col) in FAMILY_META.items():
    entries = extract_family_entries(suzuki_dict_df, family, name_col, bin_col)
    family_name_to_global[family] = {}
    for _, row in entries.iterrows():
        global_index = total_real_slots + int(row['local_index'])
        name = str(row[name_col]).strip()
        family_name_to_global[family][name] = global_index
        global_index_to_name[global_index] = name
    total_real_slots += len(entries)

# 额外预留一个“未知类别”槽位，以及每个家族一个“缺省/null”槽位。
# 这样即使原始数据某个家族为空，也能在标签向量里显式表达。
reserved_unknown_index = total_real_slots
null_condition_index = {family: total_real_slots + 1 + i for i, family in enumerate(FAMILY_META)}
class_num = total_real_slots + 1 + len(FAMILY_META)

# 记录每个家族在全局标签向量中的切片范围，后面做解码时会用到。
family_slices = {}
start = 0
for family, (name_col, bin_col) in FAMILY_META.items():
    size = len(extract_family_entries(suzuki_dict_df, family, name_col, bin_col))
    family_slices[family] = slice(start, start + size)
    start += size

def canonicalize_smiles(smiles):
    # 先解析成 RDKit 分子，再输出 canonical SMILES，确保同一分子只有一个规范写法。
    if pd.isna(smiles) or not str(smiles).strip():
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        raise ValueError(f'无法解析 SMILES: {smiles}')
    return Chem.MolToSmiles(mol, canonical=True)

def split_reaction_smiles(reaction_smiles):
    # 教学版假设一条反应由两个底物和一个产物组成：
    # Reactant1 . Reactant2 >> Product
    reactants, product = reaction_smiles.split('>>')
    reactant_parts = [part for part in reactants.split('.') if part]
    if len(reactant_parts) == 1:
        reactant_parts.append(None)
    if len(reactant_parts) != 2:
        raise ValueError(f'教学版假设两个底物，得到: {reactant_parts}')
    return {
        'Reactant1': canonicalize_smiles(reactant_parts[0]),
        'Reactant2': canonicalize_smiles(reactant_parts[1]) if reactant_parts[1] else None,
        'Product': canonicalize_smiles(product),
    }

def parse_condition_names(raw_value):
    # 原始条件列可能是“名称1|名称2|...”形式，这里切成列表。
    if pd.isna(raw_value) or not str(raw_value).strip():
        return []
    return [item.strip() for item in str(raw_value).split('|') if item.strip()]

def encode_condition_names(row):
    # 把每个家族的名称列表映射成全局标签编号列表。
    encoded = {}
    for raw_col, family in RAW_COL_TO_FAMILY.items():
        names = parse_condition_names(row[raw_col])
        indices = [family_name_to_global[family][name] for name in names]
        encoded[family] = indices
    return encoded

def encode_multihot_target(encoded_conditions):
    # 构造多标签目标向量 y。
# 公式：
# y_j = 1, 若第 j 个全局条件类别在当前反应中出现
# y_j = 0, 否则
# 若某个家族完全缺失，则把该家族专属的 null 槽位置 1
    target = np.zeros(class_num, dtype=np.float32)
    for family in FAMILY_META:
        if encoded_conditions[family]:
            target[encoded_conditions[family]] = 1.0
        else:
            target[null_condition_index[family]] = 1.0
    return target

def smiles_to_relgcn_graph(smiles):
    # 把分子转成 RelGCN 需要的图表示：
# - atom_ids: 节点原子序数
# - adj[r, i, j]: 第 r 类键是否连接 i 和 j
# - mask[i]: 第 i 个节点是否真实存在
# 公式：A^(r) in {0, 1}^{N x N}
    mol = Chem.MolFromSmiles(smiles)
    atom_ids = np.array([atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype=np.int64)
    n = len(atom_ids)
    adj = np.zeros((4, n, n), dtype=np.float32)
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        channel = BOND_TYPE_TO_CHANNEL[bond.GetBondType()]
        adj[channel, i, j] = 1.0
        adj[channel, j, i] = 1.0
    mask = np.ones(n, dtype=np.float32)
    return {'atom_ids': atom_ids, 'adj': adj, 'mask': mask, 'num_atoms': n}

# 把 reaction_smiles 拆成 Reactant1 / Reactant2 / Product 三列。
parsed_df = pd.concat([demo_df, demo_df['reaction_smiles'].apply(split_reaction_smiles).apply(pd.Series)], axis=1)

def build_item(row):
    # 每一条样本最终包含三张分子图、一个多标签向量，以及可读的条件编码结果。
    encoded = encode_condition_names(row)
    return {
        'reaction_id': row['reaction_id'],
        'graphs': [
            smiles_to_relgcn_graph(row['Reactant1']),
            smiles_to_relgcn_graph(row['Reactant2']),
            smiles_to_relgcn_graph(row['Product']),
        ],
        'labels': encode_multihot_target(encoded),
        'encoded_conditions': encoded,
    }

items = [build_item(row) for _, row in parsed_df.iterrows()]

def pad_atom_ids(atom_ids, max_atoms):
    # 把不同分子补到相同节点数，便于堆叠成 batch 张量。
    arr = np.zeros(max_atoms, dtype=np.int64)
    arr[:len(atom_ids)] = atom_ids
    return arr

def pad_mask(mask, max_atoms):
    # mask_i = 1 表示真实原子，0 表示补零得到的 padding 节点。
    arr = np.zeros(max_atoms, dtype=np.float32)
    arr[:len(mask)] = mask
    return arr

def pad_adj(adj, max_atoms):
    # 每个关系通道的邻接矩阵都补到 max_atoms x max_atoms。
    arr = np.zeros((adj.shape[0], max_atoms, max_atoms), dtype=np.float32)
    n = adj.shape[1]
    arr[:, :n, :n] = adj
    return arr

def collate(items):
    # 把若干样本整理成 batch。
    # 最终张量形状大致为：
# atom_ids_k: [B, N_k]
# adjs_k    : [B, R, N_k, N_k]
# mask_k    : [B, N_k]
# labels    : [B, C]
    batch = {'reaction_ids': [item['reaction_id'] for item in items]}
    for mol_idx in range(3):
        max_atoms = max(item['graphs'][mol_idx]['num_atoms'] for item in items)
        batch[f'atom_ids_{mol_idx+1}'] = torch.tensor(
            np.stack([pad_atom_ids(item['graphs'][mol_idx]['atom_ids'], max_atoms) for item in items]),
            dtype=torch.long,
        )
        batch[f'adjs_{mol_idx+1}'] = torch.tensor(
            np.stack([pad_adj(item['graphs'][mol_idx]['adj'], max_atoms) for item in items]),
            dtype=torch.float32,
        )
        batch[f'mask_{mol_idx+1}'] = torch.tensor(
            np.stack([pad_mask(item['graphs'][mol_idx]['mask'], max_atoms) for item in items]),
            dtype=torch.float32,
        )
    batch['labels'] = torch.tensor(np.stack([item['labels'] for item in items]), dtype=torch.float32)
    return batch

batch = collate(items)
for key, value in batch.items():
    if isinstance(value, torch.Tensor):
        print(f'{key:12s} -> shape={tuple(value.shape)}')

atom_ids_1   -> shape=(3, 9)
adjs_1       -> shape=(3, 4, 9, 9)
mask_1       -> shape=(3, 9)
atom_ids_2   -> shape=(3, 10)
adjs_2       -> shape=(3, 4, 10, 10)
mask_2       -> shape=(3, 10)
atom_ids_3   -> shape=(3, 14)
adjs_3       -> shape=(3, 4, 14, 14)
mask_3       -> shape=(3, 14)
labels       -> shape=(3, 119)


## 2. Torch 版 RelGCN 编码器

这段代码实现的是“分子内部如何传播信息”。输入还是节点级别的原子表示，输出仍然是节点级别的嵌入，只不过每个节点已经融合了邻居和键类型的信息。

原仓库的 models/relgcn.py 做了三件事：

1. 把 atom_ids 嵌入成连续向量
2. 对每一种边类型分别做消息传递
3. 在每一层之后接一个非线性变换

教学版 RelGCNLayer 不依赖 chainer-chemistry，但保留了同样的核心计算逻辑：

$$
h_i^{(l+1)} = \tanh\left(
W_0 h_i^{(l)} + \sum_r W_r \sum_j \hat{A}^{(r)}_{ij} h_j^{(l)}
\right)
$$

这里的含义是：

- h_i^{(l)} 是第 l 层时节点 i 的表示
- r 表示不同的化学键关系，例如单键、双键、三键、芳香键
- \hat{A}^{(r)} 是按关系类型归一化后的邻接矩阵
- W_0 负责节点自身信息，W_r 负责第 r 类边传来的邻居信息

因此，这个代码单元的意图不是“定义一个普通神经网络层”，而是明确告诉读者：在反应条件预测里，模型先分别理解每个分子内部的结构，再谈分子之间如何组合。

In [ ]:
# ====== Step 2: Torch 版 RelGCN ======
def rescale_relation_adjacency(adj, mask=None):
    # 对每一种关系邻接矩阵做度归一化，避免高连接度节点在消息聚合时数值过大。
    # 公式：A_hat^(r)[i, j] = A^(r)[i, j] / max(sum_j A^(r)[i, j], 1)
    degree = adj.sum(dim=1).sum(dim=-1)  # [B, N]
    degree = torch.where(degree > 0, degree, torch.ones_like(degree))
    adj = adj / degree[:, None, :, None]
    if mask is not None:
        # 对 padding 节点做边掩码，保证虚拟节点不会参与消息传递。
        edge_mask = mask[:, None, :, None] * mask[:, None, None, :]
        adj = adj * edge_mask
    return adj

class RelGCNLayer(nn.Module):
    def __init__(self, hidden_dim, num_relations=4):
        super().__init__()
        # self_linear 对应自环项 W_0 h_i，rel_linears 对应不同关系的 W_r。
        self.self_linear = nn.Linear(hidden_dim, hidden_dim)
        self.rel_linears = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(num_relations)])

    def forward(self, h, adj, mask=None):
        # 单层 RelGCN 的核心公式：
# h_i^(l+1) = tanh(W_0 h_i^(l) + sum_r W_r sum_j A_hat_ij^(r) h_j^(l))
        adj = rescale_relation_adjacency(adj, mask)
        messages = self.self_linear(h)
        for rel_id, linear in enumerate(self.rel_linears):
            neigh = torch.matmul(adj[:, rel_id], h)
            messages = messages + linear(neigh)
        out = torch.tanh(messages)
        if mask is not None:
            out = out * mask.unsqueeze(-1)
        return out

class RelGCNEncoder(nn.Module):
    def __init__(self, max_atomic_num=100, hidden_dim=64, num_layers=3, num_relations=4):
        super().__init__()
        # 先把原子序数 z 映射成向量 e_z，再堆叠多层 RelGCN。
        self.embedding = nn.Embedding(max_atomic_num + 1, hidden_dim, padding_idx=0)
        self.layers = nn.ModuleList([RelGCNLayer(hidden_dim, num_relations) for _ in range(num_layers)])

    def forward(self, atom_ids, adjs, mask):
        # h^(0) = Embedding(atom_ids)
        h = self.embedding(atom_ids)
        for layer in self.layers:
            h = layer(h, adjs, mask)
        return h

encoder_probe = RelGCNEncoder(hidden_dim=64, num_layers=3)
print('RelGCNEncoder ready:', encoder_probe.__class__.__name__, 'layers =', len(encoder_probe.layers))

RelGCNEncoder ready: RelGCNEncoder layers = 3


## 3. Reaction-level Attention Readout

这段代码回答的问题是：当三个分子都已经被编码成节点向量后，怎样把“很多个节点”压缩成“一条反应的表示”。

这是基础版 train.py 和 attention 版 train_attention_model.py 的关键差异：

- 基础版先各自做图级读出，再把 3 个分子向量直接拼接
- attention 版保留所有节点嵌入，再做跨三分子的节点级注意力聚合

原仓库 models/attention_readout.py 的思想可以拆成两步。

第一步是分子内门控。每个节点先经过一个 gate：

$$
g_i = \sigma(W_i h_i) \odot \tanh(W_j h_i)
$$

这里的 \sigma(W_i h_i) 可以理解成“这部分信息该保留多少”，而 \tanh(W_j h_i) 是节点内容本身。

第二步是反应级聚合。把三个分子的节点拼成一个更长的节点序列后，计算注意力权重和 value，再做加权求和：

$$
a_i = \sigma(W_{att} g_i), \quad
v_i = \tanh(W_{out} g_i), \quad
r = \sum_i a_i \odot v_i
$$

其中 r 就是整条反应的向量表示。

所以，这个代码单元的意图是把“节点表示”提升为“反应表示”，并且显式保留模型究竟更关注哪个分子、哪个局部结构的线索。

In [ ]:
# ====== Step 3: Torch 版 reaction attention readout ======
class ReactionAttentionReadout(nn.Module):
    def __init__(self, node_dim=64, readout_dim=128):
        super().__init__()
        # i_proj / j_proj 先对节点表示做门控投影；
        # att_proj / out_proj 再做 reaction-level attention 与 value 投影。
        self.i_proj = nn.Linear(node_dim, readout_dim)
        self.j_proj = nn.Linear(node_dim, readout_dim)
        self.att_proj = nn.Linear(readout_dim, readout_dim * 3)
        self.out_proj = nn.Linear(readout_dim, readout_dim * 3)

    def molecule_gate(self, nodes, mask):
        # 分子内节点门控：
        # g_i = sigmoid(W_i h_i) * tanh(W_j h_i)
        # 它相当于先学一个“保留多少信息”的门，再给出节点内容本身。
        gated = torch.sigmoid(self.i_proj(nodes)) * torch.tanh(self.j_proj(nodes))
        gated = gated * mask.unsqueeze(-1)
        return gated

    def forward(self, node_sets, mask_sets):
        # 先对三个分子的节点分别做门控，再把所有节点拼成一个反应级节点序列。
        gated_sets = [self.molecule_gate(nodes, mask) for nodes, mask in zip(node_sets, mask_sets)]
        concat_nodes = torch.cat(gated_sets, dim=1)
        concat_mask = torch.cat(mask_sets, dim=1)

        # reaction-level attention：
        # a_i = sigmoid(W_att g_i)
        # v_i = tanh(W_out g_i)
        # r = sum_i a_i ⊙ v_i
        # 其中 r 是整条反应的向量表示。
        att = torch.sigmoid(self.att_proj(concat_nodes))
        values = torch.tanh(self.out_proj(concat_nodes))
        weighted = att * values * concat_mask.unsqueeze(-1)
        reaction_embedding = weighted.sum(dim=1)

        # 为了可解释性，额外输出每个节点在各 attention 维度上的平均强度。
        node_attention = att.mean(dim=-1) * concat_mask
        return reaction_embedding, node_attention

readout_probe = ReactionAttentionReadout(node_dim=64, readout_dim=128)
print('ReactionAttentionReadout ready:', readout_probe.__class__.__name__)

ReactionAttentionReadout ready: ReactionAttentionReadout


## 4. 组装顶层 ARGCN 模型

前两个代码单元分别解决了两个局部问题：

- 单个分子内部如何编码
- 三个分子的节点如何汇总成一个反应向量

这一段代码要做的事情，就是把这些局部模块按 train_attention_model.py 的执行顺序真正串起来，形成一个完整的端到端预测器。

结构上可以概括为：

```text
Reactant1 graph  ┐
Reactant2 graph  ├─ shared RelGCN encoder ─┐
Product graph    ┘                         ├─ reaction attention ─ MLP ─ logits
                                           ┘
```

这里“共享编码器”非常关键，因为它表示三个分子共用同一套参数映射规则。换句话说，模型并不为底物和产物分别学三套图神经网络，而是假设它们都可以投影到同一个原子表示空间。

最终输出是每个条件类别的 logit：

$$
\text{logits} = \mathrm{MLP}(r)
$$

其中 r 是上一段 attention readout 得到的反应向量。这个 logit 还不是概率，但已经包含了模型对每个条件类别的相对偏好强度。

In [ ]:
# ====== Step 4: ARGCN 顶层模型 ======
class ARGCNPredictor(nn.Module):
    def __init__(self, max_atomic_num=100, hidden_dim=64, num_layers=3, num_relations=4, class_num=119):
        super().__init__()
        # 三个分子共享同一个 encoder。
        # 这表示模型假设“底物 1 / 底物 2 / 产物”都生活在同一种原子-键表示空间里。
        self.encoder = RelGCNEncoder(
            max_atomic_num=max_atomic_num,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            num_relations=num_relations,
        )
        self.readout = ReactionAttentionReadout(node_dim=hidden_dim, readout_dim=128)
        # 分类头把反应向量 r 映射成每个条件类别的 logit。
        # 公式：logits = MLP(r)
        self.classifier = nn.Sequential(
            nn.Linear(128 * 3, 128),
            nn.ReLU(),
            nn.Linear(128, class_num),
        )

    def forward(self, batch):
        # 对 Reactant1 / Reactant2 / Product 逐一编码。
        node_sets = []
        mask_sets = []
        for idx in (1, 2, 3):
            atom_ids = batch[f'atom_ids_{idx}']
            adjs = batch[f'adjs_{idx}']
            mask = batch[f'mask_{idx}']
            node_repr = self.encoder(atom_ids, adjs, mask)
            node_sets.append(node_repr)
            mask_sets.append(mask)
        # 反应级 attention 读出得到整条反应表示，再做多标签分类。
        reaction_embedding, node_attention = self.readout(node_sets, mask_sets)
        logits = self.classifier(reaction_embedding)
        return logits, node_attention

model_probe = ARGCNPredictor(class_num=class_num)
param_count = sum(p.numel() for p in model_probe.parameters())
print('ARGCNPredictor ready: parameters =', param_count)

ARGCNPredictor ready: parameters = 249207


## 5. 跑一次前向推理，并观察张量形状

这一段代码不训练模型，只做一件基础但非常必要的事：确认前面构造的 batch，能够被刚刚组装好的 ARGCN 正常消费。

它主要想回答三个问题：

- 输入张量的组织方式是否和模型 forward 的接口一致
- 输出的 logits 形状是否真的是 [B, C]
- 输出的 node_attention 是否真的覆盖了三个分子拼接后的全部节点

这一步虽然简单，但它承担的是“结构校验”的角色。很多 notebook 看起来代码都写完了，真正出错却往往出在这里，例如 batch 维度、padding 维度、类别数不匹配。

因此，这个代码单元的意图不是解释预测结果本身，而是先检查模型的数据流是否完整闭合。只有前向传播能跑通，后面讨论损失函数、过拟合演示和条件解码才有意义。

In [ ]:
# ====== Step 5: 前向推理演示 ======
# 固定随机种子，保证每次运行初始参数一致，便于教学演示。
torch.manual_seed(7)

model = ARGCNPredictor(class_num=class_num)
logits, node_attention = model(batch)

# 这里主要观察两个输出：
# 1. logits: 每个条件类别的未归一化分数，形状应为 [B, C]
# 2. node_attention: 拼接后每个节点的注意力强度，形状应为 [B, N_total]
print('logits shape        :', tuple(logits.shape))
print('node_attention shape:', tuple(node_attention.shape))
print('batch labels shape  :', tuple(batch['labels'].shape))
print('first sample logits (first 12 dims):')
print(logits[0, :12].detach())

logits shape        : (3, 119)
node_attention shape: (3, 33)
batch labels shape  : (3, 119)
first sample logits (first 12 dims):
tensor([-0.1848,  0.0267, -0.1473,  0.4017, -0.0912,  0.1750,  0.1544, -0.2678,
         0.1323, -0.0538,  0.5457, -0.0047])


## 6. 用极小样本做一个 Overfit 演示

为什么这里要加入一个很小的训练循环，而不是停留在随机初始化的前向输出？原因是随机 logits 基本没有可解释性，它们只能证明“程序能跑”，却不能证明“模型已经学到当前样本”。

这个代码单元的目标，是用极小样本做一次刻意的过拟合，让读者观察三件事：

1. 多标签 BCE 损失是怎样作用在条件预测上的
2. 模型是否能记住当前这几条反应
3. 预测结果怎样从概率向量解码回具体的条件名称

这里用到的损失函数是逐类别二元交叉熵。对一条样本而言，若第 j 类的 logit 是 z_j、标签是 y_j，则损失写成：

$$
\mathcal{L} = -\sum_j \left[
y_j \log \sigma(z_j) + (1-y_j) \log (1-\sigma(z_j))
\right]
$$

其中 \sigma(z_j) 表示对 logit 做 sigmoid 后得到的概率。因为这是多标签任务，不同条件类别并不是互斥关系，所以这里不能用单标签 softmax 交叉熵。

训练结束后，这段代码还会把预测按条件家族分别解码，并统计注意力在 Reactant1、Reactant2、Product 三部分的分配比例。这样你看到的不再只是一个抽象的损失值，而是“模型在记忆这几条样本时，更偏向关注哪些分子和哪些条件类别”。

In [ ]:
# ====== Step 6: tiny overfit + 条件解码 ======
torch.manual_seed(7)
model = ARGCNPredictor(class_num=class_num)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

# 这个小循环的目的不是得到泛化模型，而是检查网络能否记住这几个样本。
# BCEWithLogitsLoss 会先对 logits 做 sigmoid，再计算逐类别二元交叉熵：
# L = - sum_j [y_j log sigma(z_j) + (1 - y_j) log (1 - sigma(z_j))]
for step in range(120):
    optimizer.zero_grad()
    logits, node_attention = model(batch)
    loss = loss_fn(logits, batch['labels'])
    loss.backward()
    optimizer.step()
    if step % 20 == 0 or step == 119:
        print(f'step={step:03d}  loss={loss.item():.4f}')


def decode_predictions(prob_vector, topk=2):
    # 按条件家族分别解码预测概率。
    # 对每个家族，只在它自己的类别切片里取 top-k，避免金属和溶剂互相竞争。
    decoded = {}
    for family, family_slice in family_slices.items():
        family_probs = prob_vector[family_slice]
        top_indices = torch.topk(family_probs, k=min(topk, len(family_probs))).indices.tolist()
        decoded[family] = [
            {
                'global_index': family_slice.start + idx,
                'name': global_index_to_name[family_slice.start + idx],
                'prob': float(family_probs[idx]),
            }
            for idx in top_indices
        ]
        decoded[f'{family}_null_prob'] = float(prob_vector[null_condition_index[family]])
    return decoded


with torch.no_grad():
    # 把 logits 通过 sigmoid 转成 [0, 1] 概率。
    logits, node_attention = model(batch)
    probs = torch.sigmoid(logits)

for sample_idx, reaction_id in enumerate(batch['reaction_ids']):
    print('\n' + '=' * 80)
    print('reaction_id:', reaction_id)
    decoded = decode_predictions(probs[sample_idx], topk=2)
    for family in FAMILY_META:
        print(f'[{family}] top predictions:')
        for item in decoded[family]:
            print('  ', item)
        print(f"  null prob: {decoded[f'{family}_null_prob']:.4f}")

    # node_attention 是三个分子节点拼接后的结果。下面按节点段求和，
    # 观察模型把多少注意力质量分配给 Reactant1 / Reactant2 / Product。
    mask_lengths = [int(batch[f'mask_{idx}'][sample_idx].sum().item()) for idx in (1, 2, 3)]
    ranges = []
    start = 0
    for length in mask_lengths:
        ranges.append((start, start + length))
        start += length
    mol_names = ['Reactant1', 'Reactant2', 'Product']
    total_attention = node_attention[sample_idx].sum().item() + 1e-8
    for mol_name, (s, e) in zip(mol_names, ranges):
        score = float(node_attention[sample_idx, s:e].sum().item() / total_attention)
        print(f'  attention share - {mol_name}: {score:.3f}')

step=000  loss=0.7033


step=020  loss=0.0897


step=040  loss=0.0797
step=060  loss=0.0449


step=080  loss=0.0080
step=100  loss=0.0002


step=119  loss=0.0000

reaction_id: rxn_001
[M] top predictions:
   {'global_index': 0, 'name': 'tetrakis(triphenylphosphine) palladium(0)', 'prob': 0.999896764755249}
   {'global_index': 2, 'name': "(1,1'-bis(diphenylphosphino)ferrocene)palladium(II) dichloride", 'prob': 0.0002790411526802927}
  null prob: 0.0000
[L] top predictions:
   {'global_index': 28, 'name': 'triphenylphosphine', 'prob': 0.9997861981391907}
   {'global_index': 36, 'name': "1,1'-bis-(diphenylphosphino)ferrocene", 'prob': 0.00020023576507810503}
  null prob: 0.0000
[B] top predictions:
   {'global_index': 51, 'name': 'potassium carbonate', 'prob': 0.9999873638153076}
   {'global_index': 54, 'name': 'caesium carbonate', 'prob': 0.0007058352348394692}
  null prob: 0.0000
[S] top predictions:
   {'global_index': 86, 'name': 'tetrahydrofuran', 'prob': 0.9999855756759644}
   {'global_index': 89, 'name': '1,4-dioxane', 'prob': 0.0005785332759842277}
  null prob: 0.0000
[A] top predictions:
   {'global_index': 96, 'name

## 7. 总结

### 本 notebook 完成了什么

- 用 **Torch** 复现了 `reaction-gcnn` attention 版本的主干结构。
- 解释了为什么这个模型是“共享分子编码器 + 三分子级注意力聚合 + 多标签分类头”。
- 用一个极小样本 overfit 演示，把前向推理、损失函数和结果解码串了起来。

### 教学版与原仓库的差异

- 原仓库依赖 `chainer-chemistry` 的现成组件；教学版全部显式写成 Torch 模块。
- 原仓库还有更多训练、缓存、评估脚本；这里仅保留最核心的编码与推理路径。
- 教学版额外返回了 attention 强度，便于解释模型关注点。

到这里，你已经可以把 `reaction-gcnn` 的 attention 版本当成一个可读的 Torch 教程来理解，而不是只能黑盒调用旧代码。
